# Gradio introduction

## Learning goals

- Understand the browser → Gradio server → Python callback path
- Choose `Interface`, `ChatInterface`, or `Blocks`
- Keep callback logic pure and directly testable
- Validate user input and return safe errors
- Launch deliberately in notebooks and local browsers


## Install Gradio

For a new project:

```text
uv add gradio
```

This course repository already declares Gradio, so a learner cloning it should run `uv sync`.


## What Gradio provides

Gradio turns Python functions into a small web application. It renders browser components, serializes user input, calls your Python function on the server, and renders returned values. Gradio does not make the callback trustworthy, stateless, or safe automatically; the callback still needs validation, limits, error handling, and appropriate authentication.


## Mental model

```mermaid
sequenceDiagram
  participant B as Browser UI
  participant G as Gradio server
  participant F as Python callback
  participant S as Model or service
  B->>G: Component values
  G->>F: Python arguments
  F->>F: Validate and normalize
  opt External work
    F->>S: Bounded request
    S-->>F: Result or error
  end
  F-->>G: Component-compatible return values
  G-->>B: Updated interface
```

The callback is the application boundary. Inputs arrive from a browser and are untrusted. Exceptions should become useful, non-sensitive UI messages while detailed diagnostics go to protected logs.


## Choose the smallest Gradio abstraction

| Abstraction | Best for | Function contract |
|---|---|---|
| `gr.Interface` | One-shot input → output functions | Component values in, component values out |
| `gr.ChatInterface` | Conversational functions | Current message and chat history in, reply out |
| `gr.Blocks` | Custom layouts, events, multiple outputs, state | You wire components and events explicitly |

Start with `Interface` or `ChatInterface`. Move to `Blocks` when the product genuinely needs custom state, traces, controls, or layout—not simply because it is more flexible.


## Build and test the function before the UI

A callback should be callable like an ordinary Python function. This makes validation and unit tests independent of the browser.


In [ ]:
MAX_NAME_LENGTH = 100


def greet(name: str) -> str:
    cleaned = (name or "").strip()
    if len(cleaned) > MAX_NAME_LENGTH:
        return f"Please use at most {MAX_NAME_LENGTH} characters."
    return f"Hello, {cleaned or 'traveler'}!"


assert greet("  Asha  ") == "Hello, Asha!"
assert greet("") == "Hello, traveler!"
assert "at most" in greet("x" * 101)
print(greet("Asha"))


## Wrap it with `Interface`

Explicit components communicate labels, limits, and UX intent better than bare string shortcuts. `api_visibility="private"` prevents this teaching endpoint from being advertised as a public API; application authentication is a separate concern.


In [ ]:
import gradio as gr


greeting_demo = gr.Interface(
    fn=greet,
    inputs=gr.Textbox(label="Name", placeholder="Enter your name", max_length=MAX_NAME_LENGTH),
    outputs=gr.Textbox(label="Greeting"),
    examples=[["Asha"], ["Sanjay"]],
    title="Greeting demo",
    description="A one-shot Gradio Interface around a tested Python function.",
    api_visibility="private",
    concurrency_limit=4,
)

print(type(greeting_demo).__name__)


## `ChatInterface` adds conversation history

The callback receives the newest message and prior browser-session history. The exact history structure is UI state, not durable memory. A production app should keep server-side user data authorized and scoped rather than trusting browser history as a source of identity or permission.


In [ ]:
def echo_chat(message: str, history: list[dict[str, object]]) -> str:
    cleaned = (message or "").strip()
    if not cleaned:
        return "Please enter a message."
    if len(cleaned) > 500:
        return "Please shorten the message to 500 characters."
    return f"You said: {cleaned} (prior messages visible to UI: {len(history)})"


chat_demo = gr.ChatInterface(
    fn=echo_chat,
    title="Chat callback demo",
    description="No model call: this demonstrates the message/history contract.",
    examples=["Plan a weekend in Jaipur", "What should I pack?"],
    api_visibility="private",
    concurrency_limit=4,
)

assert echo_chat("hello", []) == "You said: hello (prior messages visible to UI: 0)"
print(type(chat_demo).__name__)


## Launch deliberately

- `inline=True` embeds the app in a compatible notebook frontend.
- `inbrowser=True` opens the local URL in your browser.
- `share=False` avoids creating a public share link.
- `show_error=False` avoids showing raw server exceptions to users.

The launch cell is disabled so automated notebook validation does not start a server.


In [ ]:
RUN_GRADIO = False

if RUN_GRADIO:
    greeting_demo.launch(
        inline=True,
        inbrowser=False,
        share=False,
        show_error=False,
    )
else:
    print("Gradio server not started. Set RUN_GRADIO = True to launch.")


## Production concerns

- Put authentication and authorization in front of sensitive callbacks.
- Set input/file limits before forwarding data to models or tools.
- Bound callback runtime, model output, concurrency, and queues.
- Keep secrets and raw stack traces out of browser responses.
- Treat uploaded filenames, MIME types, and content as untrusted.
- Use per-session state deliberately; do not store user history in process-wide globals.
- Disable public sharing for private data and understand deployment network exposure.
- Test the pure function and the browser interaction separately.


## Exercise

1. Add a travel style dropdown to `greeting_demo` and update the callback contract.
2. Test blank, whitespace-only, and over-limit input without launching Gradio.
3. Change the callback to raise an internal exception, then replace it with a safe user message.
4. Identify which state belongs in browser history and which belongs in an authenticated server store.
5. Convert the greeting demo to `Blocks` only if you can name a layout or event requirement that `Interface` cannot meet.


## Summary

Gradio is an adapter between browser components and Python functions. Start with a pure, tested callback; choose `Interface` for one-shot functions, `ChatInterface` for conversations, and `Blocks` for custom layouts or state. Validate browser input, bound expensive work, keep sessions isolated, and launch with deliberate network exposure.
